In [ ]:
# ============================================================
# PHASE 3 — TARGET CREATION
# Cell 1: Dynamic Market Refresh, Windows & Target Eligibility
# ============================================================

from datetime import date, timedelta
import pandas as pd
from src.data.providers import nse_market

TARGET_HORIZON = 252

if "recent_history" not in globals():
    raise RuntimeError("`recent_history` is not available.")

recent_history["trade_date"] = pd.to_datetime(recent_history["trade_date"], errors="coerce")

latest_before = recent_history["trade_date"].max().date()
refresh_start = latest_before + timedelta(days=1)

if refresh_start <= date.today():
    try:
        new_data = nse_market.collect_market_data(refresh_start, date.today())
    except RuntimeError:
        new_data = pd.DataFrame()

    if not new_data.empty:
        new_data["trade_date"] = pd.to_datetime(new_data["trade_date"], errors="coerce")
        recent_history = (
            pd.concat([recent_history, new_data], ignore_index=True)
            .drop_duplicates(["trade_date", "instrument_id"])
            .sort_values(["nse_symbol", "trade_date"])
            .reset_index(drop=True)
        )

latest_trade_date = recent_history["trade_date"].max()
trading_calendar = (
    recent_history["trade_date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

if len(trading_calendar) < TARGET_HORIZON:
    raise ValueError("Insufficient history for 252 trading days.")

analysis_windows = {1: "1D", 5: "5D", 20: "20D", 60: "60D", 126: "126D", 252: "252D"}
window_df = pd.DataFrame([
    {
        "window": label,
        "trading_days": days,
        "start_date": trading_calendar.iloc[-days],
        "end_date": latest_trade_date,
    }
    for days, label in analysis_windows.items()
])

target_calendar = pd.DataFrame({
    "decision_date": trading_calendar.iloc[:-TARGET_HORIZON].to_numpy(),
    "target_end_date": trading_calendar.iloc[TARGET_HORIZON:].to_numpy(),
})

print("=" * 60)
print("PHASE 3 — DYNAMIC MARKET & TARGET SETUP")
print("=" * 60)
print(f"\nLatest available trading date: {latest_trade_date.date()}")
print(f"Trading days available: {len(trading_calendar):,}")
print(f"Companies available: {recent_history['isin'].nunique():,}")
print("\nInvestment windows:")
display(window_df)
print(f"Target-eligible decision dates: {len(target_calendar):,}")
print(f"First eligible date: {target_calendar['decision_date'].min().date()}")
print(f"Last eligible date: {target_calendar['decision_date'].max().date()}")
print(f"Latest target-end date: {target_calendar['target_end_date'].max().date()}")
print(f"Current inference date: {latest_trade_date.date()}")
print("\n" + "=" * 60)
print("PHASE 3 CELL 1: PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 3 — TARGET CREATION
# Cell 2: Official NIFTY 50 NTR Benchmark Acquisition & Validation
# ============================================================

import json
import requests
import pandas as pd

history_start_date = pd.to_datetime(recent_history["trade_date"].min())
latest_trade_date = pd.to_datetime(recent_history["trade_date"].max())

endpoint = "https://www.niftyindices.com/BackPage/getTotalReturnIndexString"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json,text/javascript,*/*;q=0.9",
    "Content-Type": "application/json; charset=UTF-8",
}

start_date = history_start_date.strftime("%d/%m/%Y")
end_date = latest_trade_date.strftime("%d/%m/%Y")

request_string = (
    "{"
    f"'name':'NIFTY 50',"
    f"'startDate':'{start_date}',"
    f"'endDate':'{end_date}',"
    f"'indexName':'NIFTY 50'"
    "}"
)

response = requests.post(
    endpoint,
    headers=headers,
    json={"cinfo": request_string},
    timeout=60,
)
if response.status_code != 200:
    raise RuntimeError(f"NIFTY endpoint failed: HTTP {response.status_code}")

result = response.json()
if isinstance(result, dict) and "d" in result:
    result = result["d"]
    if isinstance(result, str):
        result = json.loads(result)

if not isinstance(result, list) or not result:
    raise RuntimeError("NIFTY endpoint returned no usable records.")

nifty_ntr = pd.DataFrame(result)
required = {"Date", "NTR_Value"}
missing = required - set(nifty_ntr.columns)
if missing:
    raise RuntimeError(f"Missing NIFTY NTR columns: {sorted(missing)}")

nifty_ntr["trade_date"] = pd.to_datetime(nifty_ntr["Date"], dayfirst=True, errors="coerce")
nifty_ntr["ntr_value"] = pd.to_numeric(nifty_ntr["NTR_Value"], errors="coerce")
nifty_ntr = (
    nifty_ntr[["trade_date", "ntr_value"]]
    .dropna()
    .drop_duplicates("trade_date")
    .sort_values("trade_date")
    .reset_index(drop=True)
)
nifty_ntr = nifty_ntr[nifty_ntr["trade_date"].between(history_start_date, latest_trade_date)].copy()

if nifty_ntr.empty or (nifty_ntr["ntr_value"] <= 0).any():
    raise RuntimeError("Invalid NIFTY NTR data.")

nifty_ntr_benchmark = nifty_ntr.copy()

market_calendar = pd.Series(
    recent_history["trade_date"].drop_duplicates().sort_values()
).reset_index(drop=True)
ntr_calendar = pd.Series(nifty_ntr["trade_date"].drop_duplicates().sort_values()).reset_index(drop=True)

missing_ntr_dates = market_calendar[~market_calendar.isin(ntr_calendar)]
current_window_start = market_calendar.iloc[-252]
current_market_dates = market_calendar[
    market_calendar.between(current_window_start, latest_trade_date)
]
current_missing = current_market_dates[~current_market_dates.isin(ntr_calendar)]
current_coverage = 1 - len(current_missing) / len(current_market_dates)

if len(current_missing) != 0:
    raise RuntimeError("NIFTY NTR does not fully cover the current 252-day window.")

print("=" * 60)
print("PHASE 3 — NIFTY 50 NTR BENCHMARK")
print("=" * 60)
print(f"\nNTR rows: {len(nifty_ntr_benchmark):,}")
print(f"NTR range: {nifty_ntr_benchmark['trade_date'].min().date()} → {nifty_ntr_benchmark['trade_date'].max().date()}")
print(f"Duplicate dates: {nifty_ntr_benchmark['trade_date'].duplicated().sum():,}")
print(f"Missing market dates: {len(missing_ntr_dates):,}")
print(f"Current-window missing: {len(current_missing):,}")
print(f"Current-window coverage: {current_coverage:.4%}")
print("\nRecent NTR values:")
print(nifty_ntr_benchmark.tail(10).to_string(index=False))
print("\n" + "=" * 60)
print("PHASE 3 CELL 2: PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 3 — TARGET CREATION
# Cell 3: 252-Day Benchmark-Relative Target Construction
# ============================================================

import numpy as np
import pandas as pd

required = {"isin", "trade_date", "close", "series"}
missing = required - set(recent_history.columns)
if missing:
    raise RuntimeError(f"Required market columns missing: {sorted(missing)}")

eq_data = recent_history[recent_history["series"].eq("EQ")].copy()
eq_data["trade_date"] = pd.to_datetime(eq_data["trade_date"], errors="coerce")
eq_data["close"] = pd.to_numeric(eq_data["close"], errors="coerce")
eq_data = (
    eq_data[["isin", "trade_date", "close"]]
    .dropna()
    .sort_values(["trade_date", "isin"])
    .reset_index(drop=True)
)

if eq_data.empty:
    raise RuntimeError("No EQ-series observations found.")
if eq_data.duplicated(["isin", "trade_date"]).any():
    raise RuntimeError("Duplicate EQ ISIN/date observations detected.")

calendar = pd.Series(
    eq_data["trade_date"].drop_duplicates().sort_values()
).reset_index(drop=True)

calendar_lookup = pd.DataFrame({
    "decision_date": calendar,
    "target_end_date": calendar.shift(-252),
}).dropna()

ntr = nifty_ntr_benchmark[["trade_date", "ntr_value"]].copy()
ntr["trade_date"] = pd.to_datetime(ntr["trade_date"])
ntr["ntr_value"] = pd.to_numeric(ntr["ntr_value"], errors="coerce")
ntr = (
    ntr.dropna()
    .drop_duplicates("trade_date")
    .sort_values("trade_date")
    .reset_index(drop=True)
)

ntr_dates = set(ntr["trade_date"])
eligible = calendar_lookup[
    calendar_lookup["decision_date"].isin(ntr_dates)
    & calendar_lookup["target_end_date"].isin(ntr_dates)
]

decision_prices = (
    eq_data.merge(
        eligible,
        left_on="trade_date",
        right_on="decision_date",
        how="inner",
    )
    .rename(columns={"close": "company_close_t"})
    [["isin", "decision_date", "target_end_date", "company_close_t"]]
)

future_prices = eq_data[
    ["isin", "trade_date", "close"]
].rename(
    columns={
        "trade_date": "target_end_date",
        "close": "company_close_t252",
    }
)

if future_prices.duplicated(["isin", "target_end_date"]).any():
    raise RuntimeError("Duplicate future company/date observations detected.")

target_data = decision_prices.merge(
    future_prices,
    on=["isin", "target_end_date"],
    how="inner",
    validate="many_to_one",
)

target_data = (
    target_data
    .merge(
        ntr.rename(columns={"trade_date": "decision_date", "ntr_value": "ntr_t"}),
        on="decision_date",
        how="inner",
    )
    .merge(
        ntr.rename(columns={"trade_date": "target_end_date", "ntr_value": "ntr_t252"}),
        on="target_end_date",
        how="inner",
    )
)

target_data["company_forward_return_252d"] = (
    target_data["company_close_t252"] / target_data["company_close_t"] - 1.0
)
target_data["nifty_forward_return_252d"] = (
    target_data["ntr_t252"] / target_data["ntr_t"] - 1.0
)
target_data["target_excess_return_252d"] = (
    target_data["company_forward_return_252d"]
    - target_data["nifty_forward_return_252d"]
)

target_data = (
    target_data
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[
            "company_forward_return_252d",
            "nifty_forward_return_252d",
            "target_excess_return_252d",
        ]
    )
    .reset_index(drop=True)
)

future_order_ok = (
    target_data["target_end_date"] > target_data["decision_date"]
).all()
duplicate_keys = target_data.duplicated(["isin", "decision_date"]).sum()
nan_count = target_data[
    ["company_forward_return_252d", "nifty_forward_return_252d", "target_excess_return_252d"]
].isna().sum().sum()
infinite_count = np.isinf(
    target_data[
        ["company_forward_return_252d", "nifty_forward_return_252d", "target_excess_return_252d"]
    ].to_numpy()
).sum()

if not future_order_ok:
    raise RuntimeError("Target leakage/order check failed.")
if duplicate_keys != 0:
    raise RuntimeError("Duplicate company/decision rows found.")
if nan_count != 0 or infinite_count != 0:
    raise RuntimeError("Invalid target values detected.")

phase3_target = target_data.copy()

print("=" * 60)
print("PHASE 3 — TARGET CONSTRUCTION")
print("=" * 60)
print(f"\nEQ rows: {len(eq_data):,}")
print(f"EQ companies: {eq_data['isin'].nunique():,}")
print(f"Final target rows: {len(phase3_target):,}")
print(f"Companies: {phase3_target['isin'].nunique():,}")
print(f"Decision dates: {phase3_target['decision_date'].nunique():,}")
print(f"Future ordering valid: {future_order_ok}")
print(f"Duplicate company/date: {duplicate_keys}")
print(f"NaN target values: {nan_count}")
print(f"Infinite target values: {infinite_count}")
print(f"Decision range: {phase3_target['decision_date'].min().date()} → {phase3_target['decision_date'].max().date()}")
print(f"Target-end range: {phase3_target['target_end_date'].min().date()} → {phase3_target['target_end_date'].max().date()}")
print("\nTarget distribution:")
print(phase3_target["target_excess_return_252d"].describe())
print("\n" + "=" * 60)
print("PHASE 3 CELL 3: PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 3 — TARGET CREATION
# Cell 4: Final Target Diagnostics & Phase 3 Completion
# ============================================================

import numpy as np
import pandas as pd

diagnostic = phase3_target.copy()
returns = diagnostic["company_forward_return_252d"]

print("=" * 60)
print("PHASE 3 — FINAL TARGET DIAGNOSTICS")
print("=" * 60)

print("\nLargest negative company returns:")
print(
    diagnostic[
        [
            "isin",
            "decision_date",
            "target_end_date",
            "company_forward_return_252d",
            "target_excess_return_252d",
        ]
    ]
    .sort_values("company_forward_return_252d")
    .head(10)
    .to_string(index=False)
)

print("\nLargest positive company returns:")
print(
    diagnostic[
        [
            "isin",
            "decision_date",
            "target_end_date",
            "company_forward_return_252d",
            "target_excess_return_252d",
        ]
    ]
    .sort_values("company_forward_return_252d", ascending=False)
    .head(10)
    .to_string(index=False)
)

tail_counts = {
    "below_-75pct": int((returns < -0.75).sum()),
    "below_-50pct": int((returns < -0.50).sum()),
    "above_+100pct": int((returns > 1.00).sum()),
    "above_+200pct": int((returns > 2.00).sum()),
    "above_+500pct": int((returns > 5.00).sum()),
}

print("\nTail counts:")
for name, count in tail_counts.items():
    print(f"{name}: {count:,}")

p01, p99 = returns.quantile([0.01, 0.99])
clipped = returns.clip(p01, p99)

print("\nTail-sensitivity diagnostic:")
print(f"1st percentile : {p01:.4f}")
print(f"99th percentile: {p99:.4f}")
print(f"Raw mean       : {returns.mean():.4f}")
print(f"Clipped mean   : {clipped.mean():.4f}")
print(f"Raw std        : {returns.std():.4f}")
print(f"Clipped std    : {clipped.std():.4f}")

final_target = phase3_target.copy()

phase3_target_metadata = {
    "horizon_trading_days": 252,
    "company_return_source": "NSE EQ close-to-close",
    "benchmark": "NIFTY 50 NTR",
    "target_definition": (
        "Company 252-day forward return minus "
        "NIFTY 50 252-day forward NTR return"
    ),
    "corporate_action_adjustment": False,
    "status": "final_for_phase_4",
    "limitation": (
        "Company prices are not fully corporate-action adjusted; "
        "extreme observations may contain mechanical price effects."
    ),
}

if final_target.empty:
    raise RuntimeError("Final target is empty.")
if final_target["target_excess_return_252d"].isna().any():
    raise RuntimeError("Final target contains NaN values.")
if np.isinf(final_target["target_excess_return_252d"]).any():
    raise RuntimeError("Final target contains infinite values.")

print("\n" + "=" * 60)
print("FINAL TARGET")
print("=" * 60)
print(f"\nTarget rows: {len(final_target):,}")
print(f"Companies: {final_target['isin'].nunique():,}")
print(f"Decision dates: {final_target['decision_date'].nunique():,}")
print(f"Decision range: {final_target['decision_date'].min().date()} → {final_target['decision_date'].max().date()}")
print(f"Target range: {final_target['target_end_date'].min().date()} → {final_target['target_end_date'].max().date()}")
print("\nTarget: target_excess_return_252d")
print("Company return: NSE EQ close-to-close")
print("Benchmark: NIFTY 50 NTR")
print("Corporate-action adjustment: Not applied")
print("Limitation: Extreme observations may contain mechanical corporate-action effects.")

print("\n" + "=" * 60)
print("PHASE 3: COMPLETE")
print("=" * 60)